# Backpropagation from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: The Value Node

Every number in our computation becomes a Value. It stores its data, its gradient, and how it was created (so it knows how to compute gradients backward).

In [ ]:
```python

class Value:

    def __init__(self, data, children=(), op=''):

        self.data = data

        self.grad = 0.0

        self._backward = lambda: None

        self._children = set(children)

        self._op = op

    def __repr__(self):

        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
```

No gradient yet (0.0). No backward function yet (no-op). The `_children` track which Values produced this one, so we can topologically sort the graph later.

### Step 2: Operations with Backward Functions

Each operation creates a new Value and defines how gradients flow backward through it.

In [ ]:
```python

def __add__(self, other):

    other = other if isinstance(other, Value) else Value(other)

    out = Value(self.data + other.data, (self, other), '+')

    def _backward():

        self.grad += out.grad

        other.grad += out.grad

    out._backward = _backward

    return out

def __mul__(self, other):

    other = other if isinstance(other, Value) else Value(other)

    out = Value(self.data * other.data, (self, other), '*')

    def _backward():

        self.grad += other.data * out.grad

        other.grad += self.data * out.grad

    out._backward = _backward

    return out

In [ ]:
```

For addition: d(a+b)/da = 1, d(a+b)/db = 1. So both inputs get the output's gradient directly.

For multiplication: d(a*b)/da = b, d(a*b)/db = a. Each input gets the other's value times the output gradient.

The `+=` is critical. A Value might be used in multiple operations. Its gradient is the sum of gradients from all paths.

### Step 3: Sigmoid and Loss

In [ ]:
```python

import math

def sigmoid(self):

    x = self.data

    x = max(-500, min(500, x))

    s = 1.0 / (1.0 + math.exp(-x))

    out = Value(s, (self,), 'sigmoid')

    def _backward():

        self.grad += (s * (1 - s)) * out.grad

    out._backward = _backward

    return out

In [ ]:
```

Sigmoid derivative: sigmoid(x) * (1 - sigmoid(x)). We computed sigmoid(x) = s during the forward pass. Reuse it. No extra work.

In [ ]:
```python

def mse_loss(predicted, target):

    diff = predicted + Value(-target)

    return diff * diff

In [ ]:
```

MSE for a single output: (predicted - target)^2. We express subtraction as addition with a negated Value.

### Step 4: Backward Pass

Topological sort ensures we process nodes in the right order -- a node's gradient is fully accumulated before we propagate through it.

In [ ]:
```python

def backward(self):

    topo = []

    visited = set()

    def build_topo(v):

        if v not in visited:

            visited.add(v)

            for child in v._children:

                build_topo(child)

            topo.append(v)

    build_topo(self)

    self.grad = 1.0

    for v in reversed(topo):

        v._backward()

In [ ]:
```

Start at the loss (gradient = 1.0, since dL/dL = 1). Walk backward through the sorted graph. Each node's `_backward` pushes gradients to its children.

### Step 5: Layer and Network

In [ ]:
```python

import random

class Neuron:

    def __init__(self, n_inputs):

        scale = (2.0 / n_inputs) ** 0.5

        self.weights = [Value(random.uniform(-scale, scale)) for _ in range(n_inputs)]

        self.bias = Value(0.0)

    def __call__(self, x):

        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)

        return act.sigmoid()

    def parameters(self):

        return self.weights + [self.bias]

class Layer:

    def __init__(self, n_inputs, n_outputs):

        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):

        out = [n(x) for n in self.neurons]

        return out[0] if len(out) == 1 else out

    def parameters(self):

        params = []

        for n in self.neurons:

            params.extend(n.parameters())

        return params

class Network:

    def __init__(self, sizes):

        self.layers = []

        for i in range(len(sizes) - 1):

            self.layers.append(Layer(sizes[i], sizes[i + 1]))

    def __call__(self, x):

        for layer in self.layers:

            x = layer(x)

            if not isinstance(x, list):

                x = [x]

        return x[0] if len(x) == 1 else x

    def parameters(self):

        params = []

        for layer in self.layers:

            params.extend(layer.parameters())

        return params

    def zero_grad(self):

        for p in self.parameters():

            p.grad = 0.0

In [ ]:
```

A Neuron takes inputs, computes weighted sum + bias, and applies sigmoid. Weight initialization scales by sqrt(2/n_inputs) to prevent sigmoid saturation in deeper networks. A Layer is a list of Neurons. A Network is a list of Layers. The `parameters()` method collects all learnable Values so we can update them.

### Step 6: Train on XOR

In [ ]:
```python

random.seed(42)

net = Network([2, 4, 1])

xor_data = [

    ([0.0, 0.0], 0.0),

    ([0.0, 1.0], 1.0),

    ([1.0, 0.0], 1.0),

    ([1.0, 1.0], 0.0),

]

learning_rate = 1.0

for epoch in range(1000):

    total_loss = Value(0.0)

    for inputs, target in xor_data:

        x = [Value(i) for i in inputs]

        pred = net(x)

        loss = mse_loss(pred, target)

        total_loss = total_loss + loss

    net.zero_grad()

    total_loss.backward()

    for p in net.parameters():

        p.data -= learning_rate * p.grad

    if epoch % 100 == 0:

        print(f"Epoch {epoch:4d} | Loss: {total_loss.data:.6f}")

print("\nXOR Results:")

for inputs, target in xor_data:

    x = [Value(i) for i in inputs]

    pred = net(x)

    print(f"  {inputs} -> {pred.data:.4f} (expected {target})")

In [ ]:
```

Watch the loss decrease. From random predictions to correct XOR outputs, driven entirely by backpropagation computing gradients and nudging weights in the right direction.

### Step 7: Circle Classification

In Lesson 02, you hand-tuned weights for circle classification. Now let the network learn them.

In [ ]:
```python

random.seed(7)

def generate_circle_data(n=100):

    data = []

    for _ in range(n):

        x1 = random.uniform(-1.5, 1.5)

        x2 = random.uniform(-1.5, 1.5)

        label = 1.0 if x1 * x1 + x2 * x2 < 1.0 else 0.0

        data.append(([x1, x2], label))

    return data

circle_data = generate_circle_data(80)

circle_net = Network([2, 8, 1])

learning_rate = 0.5

for epoch in range(2000):

    random.shuffle(circle_data)

    total_loss_val = 0.0

    for inputs, target in circle_data:

        x = [Value(i) for i in inputs]

        pred = circle_net(x)

        loss = mse_loss(pred, target)

        circle_net.zero_grad()

        loss.backward()

        for p in circle_net.parameters():

            p.data -= learning_rate * p.grad

        total_loss_val += loss.data

    if epoch % 200 == 0:

        correct = 0

        for inputs, target in circle_data:

            x = [Value(i) for i in inputs]

            pred = circle_net(x)

            predicted_class = 1.0 if pred.data > 0.5 else 0.0

            if predicted_class == target:

                correct += 1

        accuracy = correct / len(circle_data) * 100

        print(f"Epoch {epoch:4d} | Loss: {total_loss_val:.4f} | Accuracy: {accuracy:.1f}%")

In [ ]:
```

We use online SGD here -- update weights after each sample instead of accumulating the full batch. This breaks symmetry faster and avoids sigmoid saturation on the full loss landscape. Shuffling the data each epoch prevents the network from memorizing the order.

No hand-tuning. The network discovers the circular decision boundary on its own. That's the power of backpropagation: you define the architecture, the loss function, and the data. The algorithm figures out the weights.

## Exercises

In [ ]:
1. Add a `__sub__` method to the Value class (a - b = a + (-1 * b)). Then implement a `__neg__` method. Verify that the gradients are correct by comparing with manual calculation for a simple expression like (a - b)^2.

2. Add a `relu` method to Value (output max(0, x), derivative is 1 if x > 0, else 0). Replace sigmoid with relu in the hidden layers and train on XOR again. Compare convergence speed. You should see faster training -- this previews Lesson 04.

3. Implement a `__pow__` method on Value for integer powers. Use it to replace `mse_loss` with a proper `(predicted - target) ** 2` expression. Verify gradients match the original implementation.

4. Add gradient clipping to the training loop: after calling `backward()`, clip all gradients to [-1, 1]. Train a deeper network (4+ layers with sigmoid) and compare loss curves with and without clipping. This is your first defense against exploding gradients.

5. Build a visualization: after training on XOR, print the gradient of every parameter in the network. Identify which layer has the smallest gradients. This demonstrates the vanishing gradient problem you read about in the Concept section.